<a href="https://colab.research.google.com/github/nithin12342/phase2/blob/main/ml_pipeline/h5_omnifusion/notebooks/Extract_Chinese_Labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🇨🇳 Extract Chinese Dataset Labels

**Objective:** Crawl the EATD-Corpus (Chinese Dataset) folders, locate `label.txt` files, extract the depression scores, and save them to a standard CSV format for training.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Extraction Logic
We will recursively search for `*label.txt` files. 

**Assumed Format:** The file usually contains a number representing the score, or a key-value pair.

In [ ]:
import os
import glob
import pandas as pd
import re
import shutil

# ---------------------------------------------------------
# CONFIGURATION - Update this path to your Chinese Dataset folder
# ---------------------------------------------------------
dataset_root = '/content/drive/MyDrive/DAIC-WOZ_Datasets/EATD-Corpus' 
local_output_csv = '/content/eatd_labels.csv'  # Save locally first
drive_output_csv = '/content/drive/MyDrive/DAIC-WOZ_Datasets/eatd_labels.csv'
# ---------------------------------------------------------

def extract_score(file_path):
    """Try to parse the score from the text file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read().strip()
            
        # Method 1: Look for explicit integer
        # Use regex to find the first integer in the file
        match = re.search(r'\d+', content)
        if match:
            return float(match.group())
        return None
    except Exception as e:
        # print(f"Error reading {file_path}: {e}")
        return None

print(f"📂 Scanning {dataset_root} for label files...")

if not os.path.exists(dataset_root):
    print(f"❌ Error: Dataset root not found: {dataset_root}")
    print("👉 Tip: If Drive is mounted but folder is missing, check the path.")
    print("👉 Tip: If you see 'Transport endpoint' errors, restart Runtime.")
else:
    # Find all text files that look like labels
    patterns = [
        '**/label.txt',
        '**/*score.txt',
        '**/*SDS.txt'
    ]

    found_files = []
    for p in patterns:
        found_files.extend(glob.glob(os.path.join(dataset_root, p), recursive=True))

    found_files = sorted(list(set(found_files)))
    print(f"Found {len(found_files)} potential label files.")

    data = []
    for fpath in found_files:
        # Attempt to extract Participant ID from folder structure
        parent_dir = os.path.basename(os.path.dirname(fpath))
        
        # Try to find numeric ID in the folder name
        id_match = re.search(r'\d+', parent_dir)
        if id_match:
            pid = id_match.group()
        else:
            # If not in folder, check filename
            fname = os.path.basename(fpath)
            id_match = re.search(r'\d+', fname)
            pid = id_match.group() if id_match else 'Unknown'
        
        score = extract_score(fpath)
        
        if score is not None and pid != 'Unknown':
            # print(f"✅ Found: PID={pid}, Score={score} (File: {os.path.basename(fpath)})")
            data.append({'Participant_ID': pid, 'PHQ8_Score': score})
        else:
            pass # warning skipped to reduce noise

    # Create DataFrame
    if data:
        df = pd.DataFrame(data)
        # Clean duplicates
        df = df.drop_duplicates(subset=['Participant_ID'])
        
        print(f"\n🎉 Successfully extracted {len(df)} labels!")
        print(df.head())
        
        # Save LOCALLY first (Safety)
        df.to_csv(local_output_csv, index=False)
        print(f"\n💾 Saved locally to: {local_output_csv}")
        
        # Try copy to Drive
        try:
            if os.path.exists(os.path.dirname(drive_output_csv)):
                shutil.copy(local_output_csv, drive_output_csv)
                print(f"☁️  Synced to Drive: {drive_output_csv}")
            else:
                print(f"⚠️ Drive folder not found, skipping sync. Download '{local_output_csv}' manually from files tab.")
        except Exception as e:
            print(f"⚠️ Could not sync to Drive: {e}")
            print(f"👉 ACTION: Download '{local_output_csv}' from the Colab Files sidebar!")

    else:
        print("\n❌ No labels extracted. Please check the 'dataset_root' path.")

## 3. Merge with Main Labels (Optional)
If you want to append these to your main `merged_labels.csv`, run this.

In [ ]:
main_csv_path = '/content/drive/MyDrive/DAIC-WOZ_Datasets/all_labels.csv'  # Update if needed

if os.path.exists(local_output_csv) and os.path.exists(main_csv_path):
    new_labels = pd.read_csv(local_output_csv)
    main_labels = pd.read_csv(main_csv_path)
    
    print(f"Main: {len(main_labels)}, New: {len(new_labels)}")
    
    # Concatenate
    combined = pd.concat([main_labels, new_labels])
    combined = combined.drop_duplicates(subset=['Participant_ID'], keep='last')
    
    # Save backup
    output_path = main_csv_path.replace('.csv', '_updated.csv')
    combined.to_csv(output_path, index=False)
    print(f"✅ Merged: {len(combined)} total participants. Saved to {output_path}")
else:
    print("Skipping merge (files not found).")